<a href="https://colab.research.google.com/github/Mooketsi324/Dataset-for-programing-for-data-analytics/blob/main/PDAN8412_ST10114575.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")
print(df.shape)
print(df.columns.tolist())
print(df['author'].value_counts())
df.head()

(19579, 3)
['id', 'text', 'author']
author
EAP    7900
MWS    6044
HPL    5635
Name: count, dtype: int64


,id,text,author
0,id26305,"This process, however, afforded me no means of...",EAP
1,id17569,It never once occurred to me that the fumbling...,HPL
2,id11008,"In his left hand was a gold snuff box, from wh...",EAP
3,id27763,How lovely is spring As we looked from Windsor...,MWS
4,id12958,"Finding nothing else, not even gold, the Super...",HPL


Firstly, this data shows that my dataset has 19,579 rows and 3 columns. The 19,579 helps because it meets the 10,000-record minimum that it is supposed to meet. The split between the authors is extremely crucial because if one author has significantly more data than the others, the model can become lazy. In this dataset the split is roughly 7,900/6,044/5,635 across the three authors, so while EAP has a moderately larger share, the imbalance isn't severe enough to make the model lazy. The LSTM suits very well because it carries context forward and reads word by word, so it can pick up on how someone writes, which is beneficial.

**EDA PLAN:**

The main plan is to check whether there are any missing or duplicate rows. We also want to check the class balance across the three authors. We also want to check the distribution of sentence length in words per row — this matters because it'll inform how long you pad/truncate sequences to when you get to the LSTM step. We also want to have a look at the vocabulary.

**Feature Plan:**

The main column within this data structure is the text column. It has become the most relevant and most useful one. When we look at feature selection, it comes down to two decisions: how much vocabulary to keep, and how long each input sequence will be. I plan to keep the top 10,000 most frequent words and treat the rest as unknown, For sequence length, I will pad shorter sentences and trim longer ones so every input is a fixed length, based on the sentence-length distribution I find in my EDA.

**Training Plan:**

When it comes to the training, I am planning to begin with an embedding dimension of 128 and 64 LSTM units, and these are common starting points for text classification. I would also like to add a dropout rate of around 0.3 to help prevent the model from overfitting to the training data. I would like to use the Adam optimizer with categorical cross-entropy loss, since this is a 3-class classification problem. I'll train for around 10 epochs with a batch size of 64. I would split the data roughly 70/15/15 into training, validation, and test sets.


**Evaluation Plan:**

When I want to evaluate the model, I will look at overall accuracy as a baseline/starting point. But since the three authors are not perfectly balanced, I do not want to rely on accuracy by itself — it can look extremely high when the model is doing poorly on smaller classes. I will also check precision, recall, and F1-score for each author individually, to see how well the model does on each one. Lastly, I'll use a confusion matrix to see which authors the model tends to mix up with each other.


**Report Plan:**

For my report, what I want to do is structure it with an introduction covering the scenario and why I chose this dataset, followed by my EDA findings with supporting charts and numbers. From there, I will then cover my preprocessing and feature selection choices, how the model was trained (architecture and hyperparameters), and my evaluation results with what they mean. If I retrain the model, I'll explain what I changed and why, and I'll finish with a short conclusion summarising whether the model successfully achieves what the book press needed."


In [ ]:
# Check for missing values and duplicate rows
print("Missing values per column:")
print(df.isnull().sum())

print("\nNumber of duplicate rows:", df.duplicated().sum())

Missing values per column:
id        0
text      0
author    0
dtype: int64

Number of duplicate rows: 0


In [ ]:
!pip install pyspark -q

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("SpookyAuthorID").getOrCreate()

sdf = sdf = spark.read.csv("train.csv", header=True, inferSchema=True, multiLine=True, escape='"')
sdf.printSchema()
sdf.show(5)

root
 |-- id: string (nullable = true)
 |-- text: string (nullable = true)
 |-- author: string (nullable = true)

+-------+--------------------+------+
|     id|                text|author|
+-------+--------------------+------+
|id26305|This process, how...|   EAP|
|id17569|It never once occ...|   HPL|
|id11008|In his left hand ...|   EAP|
|id27763|How lovely is spr...|   MWS|
|id12958|Finding nothing e...|   HPL|
+-------+--------------------+------+
only showing top 5 rows


In [ ]:
sdf.groupBy("author").count().show()

+------+-----+
|author|count|
+------+-----+
|   MWS| 6044|
|   HPL| 5635|
|   EAP| 7900|
+------+-----+



In [ ]:
from pyspark.sql.functions import size, split

sdf = sdf.withColumn("word_count", size(split(sdf["text"], " ")))
sdf.select("word_count").describe().show()

+-------+------------------+
|summary|        word_count|
+-------+------------------+
|  count|             19579|
|   mean|26.730476530977068|
| stddev|19.048353080110637|
|    min|                 2|
|    max|               861|
+-------+------------------+



**EDA Findings:**

When I am looking at the word count summary, the average sentence length comes to 27 words, with a standard deviation of about 19 words on either side of that. When we look at the shortest sentence, it is only 2 words long, while the longest is 861 words — it shows that there is clearly an outlier compared to the rest of the data. Since most of the sentences are much shorter than that extreme, I've decided to pad/truncate my sequences to around 100 words. This should cover most sentences without wasting a lot of space padding for that rare, very long outlier.

In [ ]:
from pyspark.sql.functions import explode, col

words = sdf.select(explode(split(col("text"), " ")).alias("word"))
print("Total unique words (vocabulary size):", words.select("word").distinct().count())

Total unique words (vocabulary size): 47556


 The data shows that there are 47,556 unique words, which is my total vocabulary size. This helps confirm that my feature plan was reasonable because keeping the top 10,000 most frequent words still covers the vast majority of word usage, even though it's a smaller share of the total unique words, and that ties the number directly back to my decision.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['author'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['author'])

print("Train size:", train_df.shape)
print("Validation size:", val_df.shape)
print("Test size:", test_df.shape)

Train size: (13705, 3)
Validation size: (2937, 3)
Test size: (2937, 3)


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 10000
max_len = 100

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df['text'])

X_train = pad_sequences(tokenizer.texts_to_sequences(train_df['text']), maxlen=max_len, padding='post', truncating='post')
X_val = pad_sequences(tokenizer.texts_to_sequences(val_df['text']), maxlen=max_len, padding='post', truncating='post')
X_test = pad_sequences(tokenizer.texts_to_sequences(test_df['text']), maxlen=max_len, padding='post', truncating='post')

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

X_train shape: (13705, 100)
X_val shape: (2937, 100)
X_test shape: (2937, 100)


In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

label_encoder = LabelEncoder()
y_train = to_categorical(label_encoder.fit_transform(train_df['author']))
y_val = to_categorical(label_encoder.transform(val_df['author']))
y_test = to_categorical(label_encoder.transform(test_df['author']))

print("Classes:", label_encoder.classes_)
print("y_train shape:", y_train.shape)

Classes: ['EAP' 'HPL' 'MWS']
y_train shape: (13705, 3)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_len),
    LSTM(64),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 33s 137ms/step - accuracy: 0.4028 - loss: 1.0894 - val_accuracy: 0.4035 - val_loss: 1.0882
Epoch 2/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 28s 129ms/step - accuracy: 0.4034 - loss: 1.0884 - val_accuracy: 0.4035 - val_loss: 1.0876
Epoch 3/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 28s 131ms/step - accuracy: 0.4031 - loss: 1.0885 - val_accuracy: 0.4035 - val_loss: 1.0883
Epoch 4/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 30s 139ms/step - accuracy: 0.4042 - loss: 1.0879 - val_accuracy: 0.4035 - val_loss: 1.0878
Epoch 5/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 30s 139ms/step - accuracy: 0.4044 - loss: 1.0878 - val_accuracy: 0.4038 - val_loss: 1.0886
Epoch 6/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 41s 136ms/step - accuracy: 0.4047 - loss: 1.0861 - val_accuracy: 0.4042 - val_loss: 1.0872
Epoch 7/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 29s 133ms/step - accuracy: 0.4053 - loss: 1.0849 - val_accuracy: 0.4038 - val_loss: 1.0884
Epoch 8/10
215/215 ━━━━━━━━━━━━━━━━━━━━ 29s 133ms/step - accuracy: 0.4056 - loss: 1

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))
print(confusion_matrix(y_true, y_pred))

92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step
              precision    recall  f1-score   support

         EAP       0.82      0.82      0.82      1185
         HPL       0.82      0.81      0.81       846
         MWS       0.80      0.81      0.80       906

    accuracy                           0.81      2937
   macro avg       0.81      0.81      0.81      2937
weighted avg       0.81      0.81      0.81      2937

[[972  91 122]
 [102 685  59]
 [115  60 731]]


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(y_true), y=train_df['author'].map({c: i for i, c in enumerate(label_encoder.classes_)}))
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128),
    LSTM(64),
    Dropout(0.3),
    Dense(3, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    class_weight=class_weight_dict
)

Class weights: {0: np.float64(0.8261000602772754), 1: np.float64(1.1582995267072347), 2: np.float64(1.0797289844796345)}
Epoch 1/15
215/215 ━━━━━━━━━━━━━━━━━━━━ 33s 135ms/step - accuracy: 0.3339 - loss: 1.1000 - val_accuracy: 0.3095 - val_loss: 1.1032
Epoch 2/15
215/215 ━━━━━━━━━━━━━━━━━━━━ 29s 134ms/step - accuracy: 0.3432 - loss: 1.0990 - val_accuracy: 0.4042 - val_loss: 1.0954
Epoch 3/15
215/215 ━━━━━━━━━━━━━━━━━━━━ 42s 140ms/step - accuracy: 0.3403 - loss: 1.0990 - val_accuracy: 0.2932 - val_loss: 1.0998
Epoch 4/15
215/215 ━━━━━━━━━━━━━━━━━━━━ 30s 139ms/step - accuracy: 0.3240 - loss: 1.0982 - val_accuracy: 0.3102 - val_loss: 1.0977
Epoch 5/15
215/215 ━━━━━━━━━━━━━━━━━━━━ 29s 136ms/step - accuracy: 0.3359 - loss: 1.0976 - val_accuracy: 0.3112 - val_loss: 1.1010
Epoch 6/15
215/215 ━━━━━━━━━━━━━━━━━━━━ 30s 138ms/step - accuracy: 0.3421 - loss: 1.0955 - val_accuracy: 0.2928 - val_loss: 1.0999
Epoch 7/15
215/215 ━━━━━━━━━━━━━━━━━━━━ 30s 139ms/step - accuracy: 0.3356 - loss: 1.0958 - va

When it comes to my first model, which only reached 40% accuracy, the confusion matrix displayed that it was almost highly prediciting EAP, since the EAP is the largest class in the dataset. it had taught itself that to guess the majority class rather than actually learning each author's writing style, especially failing completely on HPL. To fix this, I retrained the model using class weights, which penalize mistakes on the smaller classes more heavily, and that increased training from 10 to 15 epochs to give it more time to learn. After the retraining has been done, accuracy improved to 81%, with precision and recall fairly balanced across all three authors, showing now that the model is now distinguishing between authors rather than defaulting to one. What I did notice is some overfitting by the final epochs, where training accuracy reaches 97% while validation accurary stayed around 79%